In [17]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

In [18]:
load_dotenv()
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)

In [19]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [20]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [21]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation': response}

In [22]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [24]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!\n\n(get it?)",
 'explanation': 'The joke relies on a play on words, specifically a pun. A "crusty" pizza has a double meaning here: the crust is a literal part of a pizza, the bread-like exterior surrounding the filling. However, "crusty" also describes someone or something that is irritable or in a bad mood. The joke uses this wordplay to create a humorous connection between the pizza\'s crust and its emotional state.'}

In [26]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'chicken'}, config=config2)

{'topic': 'chicken',
 'joke': "Here's one:\n\nWhy did the chicken go to the doctor?\n\nBecause it had fowl breath! (get it?)",
 'explanation': 'This joke is an example of a play on words, which is a common technique used in puns. In this case, the punchline "fowl breath" is a pun on the phrase "foul breath," which refers to bad or unpleasant breath. However, in this joke, "fowl" is used instead of "foul," referencing the fact that a chicken is a type of bird known as a "fowl." This wordplay creates a clever connection between the setup of the joke and the punchline, making the joke humorous.'}

In [27]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'chicken', 'joke': "Here's one:\n\nWhy did the chicken go to the doctor?\n\nBecause it had fowl breath! (get it?)", 'explanation': 'This joke is an example of a play on words, which is a common technique used in puns. In this case, the punchline "fowl breath" is a pun on the phrase "foul breath," which refers to bad or unpleasant breath. However, in this joke, "fowl" is used instead of "foul," referencing the fact that a chicken is a type of bird known as a "fowl." This wordplay creates a clever connection between the setup of the joke and the punchline, making the joke humorous.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1b8d4f-966b-6592-8002-69b17ea15079'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-25T11:33:48.203150+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1b8d4f-02be-63c3-8001-c0b30b70c754'}}, tasks=(), inter

In [28]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!\n\n(get it?)", 'explanation': 'The joke relies on a play on words, specifically a pun. A "crusty" pizza has a double meaning here: the crust is a literal part of a pizza, the bread-like exterior surrounding the filling. However, "crusty" also describes someone or something that is irritable or in a bad mood. The joke uses this wordplay to create a humorous connection between the pizza\'s crust and its emotional state.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b8d49-7dc9-6533-8004-96ac7d6abe9b'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-09-25T11:31:04.558929+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b8d49-1a55-6e87-8003-9e3dcd72329c'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'jo